In [ ]:

from __future__ import annotations

import argparse
import calendar
from pathlib import Path

import numpy as np
import pandas as pd


METHODS = ["LH", "UKIH", "Fixed", "Slide", "Local", "Eckhardt"]
MIN_DAILY_COVERAGE = 0.90
MIN_VALID_METHODS = 4

def lyne_hollick(q: np.ndarray, beta: float = 0.925) -> np.ndarray:
    """Two-pass Lyne-Hollick baseflow filter."""
    q = np.asarray(q, dtype=float)
    b = np.zeros_like(q, dtype=float)

    b[0] = q[0]

    # Forward pass
    for i in range(len(q) - 1):
        b[i + 1] = beta * b[i] + (1.0 - beta) / 2.0 * (q[i] + q[i + 1])
        if b[i + 1] > q[i + 1]:
            b[i + 1] = q[i + 1]

    # Backward pass
    b_forward = b.copy()

    for i in range(len(q) - 2, -1, -1):
        b[i] = (
            beta * b[i + 1]
            + (1.0 - beta) / 2.0 * (b_forward[i + 1] + b_forward[i])
        )
        if b[i] > b_forward[i]:
            b[i] = b_forward[i]

    return np.clip(b, 0.0, q)


def _linear_interpolation(q: np.ndarray, turning_points: np.ndarray) -> np.ndarray:
    """Interpolate baseflow between turning points, bounded by streamflow."""
    q = np.asarray(q, dtype=float)
    turning_points = np.asarray(turning_points, dtype=int)

    if len(turning_points) < 2:
        raise ValueError("At least two turning points are required.")

    b = np.zeros_like(q, dtype=float)
    segment = 0

    for i in range(turning_points[0], turning_points[-1] + 1):
        if i == turning_points[segment + 1]:
            segment += 1
            b[i] = q[i]
        else:
            left = turning_points[segment]
            right = turning_points[segment + 1]

            b[i] = q[left] + (
                (q[right] - q[left])
                / (right - left)
                * (i - left)
            )

        if b[i] > q[i]:
            b[i] = q[i]

    return b

# UKIH


def ukih(q: np.ndarray, b_lh: np.ndarray) -> np.ndarray:
    """UK Institute of Hydrology smooth-minima method."""
    q = np.asarray(q, dtype=float)

    block_length = 5
    block_end = len(q) // block_length * block_length

    if block_end < 15:
        raise ValueError("Series is too short for UKIH.")

    minima = np.argmin(
        q[:block_end].reshape(-1, block_length),
        axis=1,
    )

    minima = minima + np.arange(
        0,
        block_end,
        block_length,
    )

    turning = np.zeros(len(minima), dtype=int)

    for i in range(len(minima) - 2):
        middle = minima[i + 1]

        if (
            0.9 * q[middle] < q[minima[i]]
            and 0.9 * q[middle] < q[minima[i + 2]]
        ):
            turning[i] = middle

    turning = turning[turning != 0]

    if len(turning) < 3:
        raise ValueError("UKIH found fewer than three turning points.")

    b = _linear_interpolation(q, turning)

    # Use LH at the beginning and end, as in the package implementation.
    b[: turning[0]] = b_lh[: turning[0]]
    b[turning[-1] + 1 :] = b_lh[turning[-1] + 1 :]

    return np.clip(b, 0.0, q)

# HYSEP


def hysep_interval(area_km2: float | None = None) -> int:
    """
    HYSEP interval.

    The empirical relationship uses drainage area converted from km2
    to square miles. Without area metadata, N = 5 is used, matching
    the fallback in the baseflow implementation.
    """
    if area_km2 is None or not np.isfinite(area_km2):
        n = 5.0
    else:
        if area_km2 <= 0:
            raise ValueError("Drainage area must be positive.")
        n = (0.3861022 * area_km2) ** 0.2

    interval = int(np.ceil(2.0 * n))

    if interval % 2 == 0:
        interval -= 1

    return int(min(max(interval, 3), 11))


def hysep_fixed(q: np.ndarray, area_km2: float | None = None) -> np.ndarray:
    """HYSEP fixed-interval method."""
    q = np.asarray(q, dtype=float)
    interval = hysep_interval(area_km2)

    b = np.zeros_like(q, dtype=float)
    nblocks = len(q) // interval

    for i in range(nblocks):
        start = interval * i
        stop = interval * (i + 1)
        b[start:stop] = np.min(q[start:stop])

    if nblocks * interval != len(q):
        b[nblocks * interval :] = np.min(q[nblocks * interval :])

    return np.clip(b, 0.0, q)


def hysep_sliding(q: np.ndarray, area_km2: float | None = None) -> np.ndarray:
    """HYSEP sliding-interval method."""
    q = np.asarray(q, dtype=float)
    interval = hysep_interval(area_km2)
    half = (interval - 1) // 2

    b = np.zeros_like(q, dtype=float)

    for i in range(half, len(q) - half):
        b[i] = np.min(q[i - half : i + half + 1])

    if half > 0:
        b[:half] = np.min(q[:half])
        b[len(q) - half :] = np.min(q[len(q) - half :])

    return np.clip(b, 0.0, q)


def hysep_local(
    q: np.ndarray,
    b_lh: np.ndarray,
    area_km2: float | None = None,
) -> np.ndarray:
    """HYSEP local-minimum method."""
    q = np.asarray(q, dtype=float)

    interval = hysep_interval(area_km2)
    half = (interval - 1) // 2

    turning = []

    for i in range(half, len(q) - half):
        window = q[i - half : i + half + 1]

        if q[i] == np.min(window):
            turning.append(i)

    turning = np.asarray(turning, dtype=int)

    if len(turning) < 3:
        raise ValueError("HYSEP local-minimum found fewer than three turning points.")

    b = _linear_interpolation(q, turning)

    b[: turning[0]] = b_lh[: turning[0]]
    b[turning[-1] + 1 :] = b_lh[turning[-1] + 1 :]

    return np.clip(b, 0.0, q)

# ECKHARDT PARAMETER ESTIMATION


def strict_baseflow_mask(q: np.ndarray) -> np.ndarray:
    """Identify recession/baseflow points used for recession estimation."""
    q = np.asarray(q, dtype=float)

    if len(q) < 10:
        raise ValueError("Series is too short for recession analysis.")

    dq = (q[2:] - q[:-2]) / 2.0

    # Positive/zero derivative points
    wet1 = np.concatenate(
        [[True], dq >= 0, [True]]
    )

    transitions = (
        wet1[1:].astype(int)
        - wet1[:-1].astype(int)
    )

    idx_first = np.where(transitions == 1)[0] + 1
    idx_last = np.where(transitions == -1)[0]

    idx_before = []
    for idx in idx_first:
        idx_before.extend([idx - 1, idx - 2])

    idx_after_transition = []
    for idx in idx_last:
        idx_after_transition.extend([idx + 1, idx + 2, idx + 3])

    wet2 = np.zeros(len(q), dtype=bool)

    remove = np.asarray(
        idx_before + idx_after_transition,
        dtype=int,
    )

    if len(remove):
        wet2[np.clip(remove, 0, len(q) - 1)] = True

    # Five days after major events
    growing = np.concatenate(
        [[True], (q[1:] - q[:-1]) >= 0, [True]]
    )

    idx_major = np.where(
        (q >= np.quantile(q, 0.9))
        & growing[:-1]
        & ~growing[1:]
    )[0]

    wet3 = np.zeros(len(q), dtype=bool)

    after_major = []

    for idx in idx_major:
        after_major.extend(
            [idx + 1, idx + 2, idx + 3, idx + 4, idx + 5]
        )

    if after_major:
        wet3[
            np.clip(
                np.asarray(after_major, dtype=int),
                0,
                len(q) - 1,
            )
        ] = True

    # Points followed by a larger -dQ/dt
    wet4 = np.concatenate(
        [[True], dq[1:] - dq[:-1] < 0, [True, True]]
    )

    return ~(wet1 | wet2 | wet3 | wet4)


def recession_coefficient(q: np.ndarray, strict: np.ndarray) -> float:
    """Estimate the station-specific recession coefficient a."""
    q = np.asarray(q, dtype=float)
    strict = np.asarray(strict, dtype=bool)

    cq = q[1:-1]
    dq = (q[2:] - q[:-2]) / 2.0

    cq = cq[strict[1:-1]]
    dq = dq[strict[1:-1]]

    valid = (
        np.isfinite(cq)
        & np.isfinite(dq)
        & (cq > 0)
        & (dq < 0)
    )

    cq = cq[valid]
    dq = dq[valid]

    if len(cq) < 5:
        raise ValueError(
            "Insufficient recession points for Eckhardt parameter estimation."
        )

    ratio = -dq / cq

    idx = np.argsort(ratio)[
        int(np.floor(len(ratio) * 0.05))
    ]

    k = -cq[idx] / dq[idx]

    if not np.isfinite(k) or k <= 0:
        raise ValueError("Invalid recession constant.")

    a = np.exp(-1.0 / k)

    if not (0.0 < a < 1.0):
        raise ValueError("Estimated recession coefficient is outside (0, 1).")

    return float(a)


def _moving_average(x: np.ndarray, window: int) -> np.ndarray:
    result = np.convolve(
        x,
        np.ones(window),
    ) / window

    return result[
        window - 1 : -window + 1
    ]


def recession_period_indices(q: np.ndarray) -> np.ndarray:
    """Indices used to calibrate Eckhardt BFImax."""
    q = np.asarray(q, dtype=float)

    decreasing = np.zeros(
        len(q) - 1,
        dtype=np.int64,
    )

    q_average = _moving_average(q, 3)

    decreasing[1:-1] = (
        q_average[:-1]
        - q_average[1:]
    ) > 0

    starts = np.where(
        decreasing[:-1]
        - decreasing[1:]
        == -1
    )[0] + 1

    stops = np.where(
        decreasing[:-1]
        - decreasing[1:]
        == 1
    )[0] + 1

    # Pair valid recession starts and stops.
    n = min(len(starts), len(stops))

    starts = starts[:n]
    stops = stops[:n]

    keep = (stops - starts) >= 10

    starts = starts[keep]
    stops = stops[keep]

    duration = stops - starts

    starts = (
        starts
        + np.ceil(duration * 0.6).astype(int)
    )

    pieces = [
        np.arange(start, stop, dtype=int)
        for start, stop in zip(starts, stops)
        if stop > start
    ]

    if not pieces:
        return np.asarray([], dtype=int)

    return np.concatenate(pieces)


def eckhardt(
    q: np.ndarray,
    b_lh: np.ndarray,
    a: float,
    bfi_max: float,
    return_exceed: bool = False,
) -> np.ndarray:
    """Eckhardt recursive digital filter."""
    q = np.asarray(q, dtype=float)

    size = (
        len(q) + 1
        if return_exceed
        else len(q)
    )

    b = np.zeros(size, dtype=float)
    b[0] = b_lh[0]

    for i in range(len(q) - 1):
        numerator = (
            (1.0 - bfi_max) * a * b[i]
            + (1.0 - a) * bfi_max * q[i + 1]
        )

        denominator = (
            1.0
            - a * bfi_max
        )

        b[i + 1] = numerator / denominator

        if b[i + 1] > q[i + 1]:
            b[i + 1] = q[i + 1]

            if return_exceed:
                b[-1] += 1

    return b


def calibrate_eckhardt_bfimax(
    q: np.ndarray,
    b_lh: np.ndarray,
    a: float,
) -> float:
    """
    Calibrate BFImax over 0.001...0.999 using the objective
    used by the Xie et al. baseflow workflow.
    """
    q = np.asarray(q, dtype=float)

    recession_idx = recession_period_indices(q)

    if len(recession_idx) == 0:
        raise ValueError(
            "No sufficiently long recession periods found for BFImax calibration."
        )

    other = np.ones(
        len(q),
        dtype=bool,
    )

    other[recession_idx] = False

    log_q = np.log1p(q)

    best_parameter = None
    best_loss = np.inf

    for parameter in np.arange(
        0.001,
        1.0,
        0.001,
    ):
        estimate = eckhardt(
            q,
            b_lh,
            a,
            float(parameter),
            return_exceed=True,
        )

        exceed_frequency = (
            estimate[-1]
            / len(q)
        )

        log_b = np.log1p(
            estimate[:-1]
        )

        # NSE for recession periods
        obs = log_q[recession_idx]
        sim = log_b[recession_idx]

        ss_res = np.sum(
            (obs - sim) ** 2
        )

        ss_tot = np.sum(
            (obs - np.mean(obs)) ** 2
        )

        nse_recession = (
            1.0
            - ss_res / (ss_tot + 1e-10)
            - 1e-10
        )

        # NSE for remaining periods
        obs = log_q[other]
        sim = log_b[other]

        ss_res = np.sum(
            (obs - sim) ** 2
        )

        ss_tot = np.sum(
            (obs - np.mean(obs)) ** 2
        )

        nse_other = (
            1.0
            - ss_res / (ss_tot + 1e-10)
            - 1e-10
        )

        loss = (
            1.0
            - (
                1.0
                - (1.0 - nse_recession)
                / (1.0 - nse_other)
            )
            * (1.0 - exceed_frequency)
        )

        if loss < best_loss:
            best_loss = loss
            best_parameter = float(parameter)

    if best_parameter is None:
        raise ValueError("Eckhardt BFImax calibration failed.")

    return best_parameter

# STATION SEPARATION


def separate_six_methods(
    q: np.ndarray,
    area_km2: float | None = None,
):
    """Run all six methods for one continuous daily series."""
    q = np.asarray(q, dtype=float)

    if len(q) < 30:
        raise ValueError("At least 30 daily observations are required.")

    if not np.all(np.isfinite(q)):
        raise ValueError(
            "Non-finite daily streamflow remains after preprocessing."
        )

    if np.any(q < 0):
        raise ValueError("Negative discharge values are not permitted.")

    b_lh = lyne_hollick(q)

    strict = strict_baseflow_mask(q)

    a = recession_coefficient(
        q,
        strict,
    )

    bfi_max = calibrate_eckhardt_bfimax(
        q,
        b_lh,
        a,
    )

    outputs = {
        "LH": b_lh,
        "UKIH": ukih(q, b_lh),
        "Fixed": hysep_fixed(q, area_km2),
        "Slide": hysep_sliding(q, area_km2),
        "Local": hysep_local(q, b_lh, area_km2),
        "Eckhardt": eckhardt(
            q,
            b_lh,
            a,
            bfi_max,
        ),
    }

    for method, values in outputs.items():
        values = np.asarray(values, dtype=float)

        if len(values) != len(q):
            raise ValueError(
                f"{method} returned the wrong number of values."
            )

        if not np.all(np.isfinite(values)):
            raise ValueError(
                f"{method} produced non-finite baseflow values."
            )

        if np.any(values < -1e-12):
            raise ValueError(
                f"{method} produced negative baseflow."
            )

        if np.any(values - q > 1e-10):
            raise ValueError(
                f"{method} produced baseflow greater than streamflow."
            )

        outputs[method] = np.clip(
            values,
            0.0,
            q,
        )

    return outputs, a, bfi_max

# MONTHLY ENSEMBLE


def monthly_ensemble(
    dates: pd.DatetimeIndex,
    q: np.ndarray,
    method_outputs: dict[str, np.ndarray],
    min_daily_coverage: float = MIN_DAILY_COVERAGE,
    min_valid_methods: int = MIN_VALID_METHODS,
) -> pd.DataFrame:
    """Aggregate daily outputs and create the six-method monthly median."""
    frame = pd.DataFrame(
        {
            "time": pd.to_datetime(dates),
            "Q": q,
        }
    )

    for method in METHODS:
        frame[method] = method_outputs[method]

    frame["year"] = frame["time"].dt.year
    frame["month"] = frame["time"].dt.month

    rows = []

    for (year, month), group in frame.groupby(
        ["year", "month"],
        sort=True,
    ):
        expected_days = calendar.monthrange(
            int(year),
            int(month),
        )[1]

        valid_q_days = int(
            group["Q"].notna().sum()
        )

        coverage = (
            valid_q_days
            / expected_days
        )

        row = {
            "time": pd.Timestamp(
                year=int(year),
                month=int(month),
                day=1,
            ),
            "year": int(year),
            "month": int(month),
            "expected_days": expected_days,
            "valid_Q_days": valid_q_days,
            "daily_coverage": coverage,
        }

        if coverage < min_daily_coverage:
            row["Q_monthly"] = np.nan

            for method in METHODS:
                row[f"Qb_{method}"] = np.nan

            row["n_valid_methods"] = 0
            row["Qb_ensemble"] = np.nan
            row["quickflow_ensemble"] = np.nan

            rows.append(row)
            continue

        q_month = float(
            group["Q"].sum()
        )

        row["Q_monthly"] = q_month

        monthly_methods = []

        for method in METHODS:
            valid_days = int(
                group[method].notna().sum()
            )

            method_coverage = (
                valid_days
                / expected_days
            )

            if method_coverage >= min_daily_coverage:
                value = float(
                    group[method].sum()
                )

                value = float(
                    np.clip(
                        value,
                        0.0,
                        q_month,
                    )
                )
            else:
                value = np.nan

            row[
                f"Qb_{method}"
            ] = value

            if np.isfinite(value):
                monthly_methods.append(
                    value
                )

        row["n_valid_methods"] = len(
            monthly_methods
        )

        if len(monthly_methods) >= min_valid_methods:
            ensemble = float(
                np.median(
                    monthly_methods
                )
            )

            ensemble = float(
                np.clip(
                    ensemble,
                    0.0,
                    q_month,
                )
            )

            row["Qb_ensemble"] = ensemble
            row["quickflow_ensemble"] = (
                q_month
                - ensemble
            )
        else:
            row["Qb_ensemble"] = np.nan
            row["quickflow_ensemble"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)


# INPUT / OUTPUT


def read_area_metadata(path: str | None) -> dict[str, float]:
    """Read optional station-area metadata."""
    if path is None:
        return {}

    metadata = pd.read_csv(path)

    required = {
        "station",
        "area_km2",
    }

    missing = required - set(
        metadata.columns
    )

    if missing:
        raise ValueError(
            "Area metadata must contain columns: station, area_km2"
        )

    area = {}

    for _, row in metadata.iterrows():
        station = str(row["station"])
        value = pd.to_numeric(
            row["area_km2"],
            errors="coerce",
        )

        if np.isfinite(value) and value > 0:
            area[station] = float(value)

    return area


def process_file(
    input_csv: str,
    output_dir: str,
    area_csv: str | None = None,
):
    """Process a wide daily-flow CSV."""
    input_path = Path(input_csv)
    output_path = Path(output_dir)
    output_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    raw = pd.read_csv(
        input_path
    )

    if "time" not in raw.columns:
        raise ValueError(
            "Input CSV must contain a 'time' column."
        )

    raw["time"] = pd.to_datetime(
        raw["time"],
        errors="coerce",
    )

    if raw["time"].isna().any():
        raise ValueError(
            "One or more dates could not be parsed."
        )

    if raw["time"].duplicated().any():
        raise ValueError(
            "Duplicate dates are present."
        )

    raw = raw.sort_values(
        "time"
    ).reset_index(drop=True)

    expected_index = pd.date_range(
        raw["time"].min(),
        raw["time"].max(),
        freq="D",
    )

    if len(expected_index) != len(raw):
        raise ValueError(
            "The input example is not a continuous daily date sequence. "
            "Insert explicit missing dates before applying the workflow."
        )

    if not np.array_equal(
        raw["time"].to_numpy(
            dtype="datetime64[ns]"
        ),
        expected_index.to_numpy(
            dtype="datetime64[ns]"
        ),
    ):
        raise ValueError(
            "The time column is not a complete daily sequence."
        )

    station_columns = [
        col
        for col in raw.columns
        if col != "time"
    ]

    if not station_columns:
        raise ValueError(
            "No station columns were found."
        )

    areas = read_area_metadata(
        area_csv
    )

    all_monthly = []
    summary = []

    for station in station_columns:
        q = pd.to_numeric(
            raw[station],
            errors="coerce",
        )

        valid_count = int(
            q.notna().sum()
        )

        coverage_all = (
            valid_count
            / len(q)
        )

        if q.isna().any():
            raise ValueError(
                f"{station}: the supplied GitHub sample contains a missing "
                "daily discharge value. This script intentionally does not "
                "interpolate streamflow silently."
            )

        q = q.to_numpy(
            dtype=float
        )

        if np.any(q < 0):
            raise ValueError(
                f"{station}: negative discharge detected."
            )

        area = areas.get(
            station
        )

        outputs, a, bfi_max = separate_six_methods(
            q,
            area_km2=area,
        )

        monthly = monthly_ensemble(
            pd.DatetimeIndex(
                raw["time"]
            ),
            q,
            outputs,
        )

        monthly.insert(
            0,
            "station",
            station,
        )

        valid_month = (
            monthly["Q_monthly"].notna()
            & monthly["Qb_ensemble"].notna()
            & (monthly["Q_monthly"] > 0)
        )

        denominator = monthly.loc[
            valid_month,
            "Q_monthly",
        ].sum()

        if denominator <= 0:
            bfi = np.nan
        else:
            bfi = (
                monthly.loc[
                    valid_month,
                    "Qb_ensemble",
                ].sum()
                / denominator
            )

        summary.append(
            {
                "station": station,
                "area_km2": area,
                "HYSEP_interval_days": hysep_interval(area),
                "recession_coefficient_a": a,
                "Eckhardt_BFImax": bfi_max,
                "BFI_ensemble": bfi,
                "daily_records": len(q),
                "daily_coverage": coverage_all,
                "monthly_records": len(monthly),
                "valid_months": int(valid_month.sum()),
                "minimum_valid_methods_per_month": int(
                    monthly[
                        "n_valid_methods"
                    ].min()
                ),
            }
        )

        all_monthly.append(
            monthly
        )

    monthly_out = pd.concat(
        all_monthly,
        ignore_index=True,
    )

    summary_out = pd.DataFrame(
        summary
    )

    monthly_file = (
        output_path
        / "six_method_monthly_ensemble.csv"
    )

    summary_file = (
        output_path
        / "six_method_BFI_summary.csv"
    )

    monthly_out.to_csv(
        monthly_file,
        index=False,
    )

    summary_out.to_csv(
        summary_file,
        index=False,
    )

    print(
        "\nSix-method BFI workflow completed successfully."
    )

    print(
        f"Stations processed: {len(summary_out)}"
    )

    print(
        f"Monthly output: {monthly_file}"
    )

    print(
        f"BFI summary: {summary_file}\n"
    )

    print(
        summary_out.to_string(
            index=False
        )
    )

    return summary_out, monthly_out


def main():
    parser = argparse.ArgumentParser(
        description="Six-method monthly median BFI ensemble."
    )

    parser.add_argument(
        "input_csv",
        help="Wide daily streamflow CSV with time + station columns.",
    )

    parser.add_argument(
        "--output-dir",
        default="results",
        help="Output directory.",
    )

    parser.add_argument(
        "--area-csv",
        default=None,
        help="Optional CSV containing station,area_km2.",
    )

    args = parser.parse_args()

    process_file(
        input_csv=args.input_csv,
        output_dir=args.output_dir,
        area_csv=args.area_csv,
    )


if __name__ == "__main__":
    main()
